In [0]:
pip install shap

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.statespace.sarimax import SARIMAXResults, SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import warnings
import joblib
import xgboost as xgb
from scipy import stats
import shap
from sklearn.ensemble import VotingRegressor, RandomForestRegressor
import seaborn as sns

### Define functions

In [0]:
def evaluate(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    # MAPE alleen waar y_true > 0 (technische bescherming)
    mask = y_true > 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / 
                           y_true[mask])) * 100

    print(f"Metrics {label}")
    print(f"  MAE   : €{mae:,.0f}")
    print(f"  RMSE  : €{rmse:,.0f}")
    print(f"  MAPE  : {mape:.1f}%")
    print(f"  R2    : {r2:.3f}")
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}

In [0]:
def plot_predictions(y_test, y_pred, model_name='SARIMAX'):

    mask = y_test > 0
    mape = np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100
    
    # make plot
    fig, ax = plt.subplots(figsize=(14, 6))

    # True data
    ax.plot(y_test.index, y_test.values, label='True',
            color='steelblue', linewidth=0.8)
    
    # Prediction
    ax.plot(y_pred.index, y_pred.values, label=f'Prediction {model_name}',
            color='darkorange', linewidth=0.8, linestyle='--')

    # Layout
    ax.set_title(f'{model_name} Test 2025 | MAPE: {mape:.1f}%', fontsize=12)
    ax.set_ylabel('Revenue (€)')
    ax.legend()

    # Set X-as to month
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

    plt.tight_layout()
    plt.show()

In [0]:
def cross_validate_sarimax(y_train, order, seasonal_order, X_train=None, n_splits=4):
    """
    Expanding Window Cross-Validatie for a SARIMAX-model.
    Calculates en prints mean +- std
    """
    warnings.filterwarnings('ignore', category=UserWarning, module='statsmodels')

    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    cv_scores = {
        'MAE': [],
        'RMSE': [],
        'MAPE': [],
        'R2': []
    }
    
    for fold, (train_index, test_index) in enumerate(tscv.split(y_train)):
        y_train_cv, y_test_cv = y_train.iloc[train_index], y_train.iloc[test_index]
        
        X_train_cv = X_train.iloc[train_index] if X_train is not None else None
        X_test_cv = X_train.iloc[test_index] if X_train is not None else None
        
        model_cv = SARIMAX(
            endog=y_train_cv,
            exog=X_train_cv,
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        fitted_model_cv = model_cv.fit(disp=False)
        
        # Predict
        if X_train is not None:
            pred_cv = fitted_model_cv.forecast(steps=len(y_test_cv), exog=X_test_cv)
        else:
            pred_cv = fitted_model_cv.forecast(steps=len(y_test_cv))
        
        metrics = evaluate(y_test_cv.values, pred_cv.values, label=f'CV Fold {fold + 1}')
        
        # save scores
        for key in cv_scores.keys():
            cv_scores[key].append(metrics[key])
            
    # Calculate statistics
    cv_results = {
        'mean': {key: np.mean(values) for key, values in cv_scores.items()},
        'std': {key: np.std(values) for key, values in cv_scores.items()}
    }
    
    # Print results
    print(f"SARIMAX Cross-Validation Scores ({n_splits} Folds) ---")
    print(f"  MAE   : €{cv_results['mean']['MAE']:,.0f}  (± €{cv_results['std']['MAE']:,.0f})")
    print(f"  RMSE  : €{cv_results['mean']['RMSE']:,.0f}  (± €{cv_results['std']['RMSE']:,.0f})")
    print(f"  MAPE  : {cv_results['mean']['MAPE']:.1f}%  (± {cv_results['std']['MAPE']:.1f}%)")
    print(f"  R2    : {cv_results['mean']['R2']:.3f}  (± {cv_results['std']['R2']:.3f})")
    
    return cv_scores

In [0]:
def cross_validate_ml(model, X, y, n_splits=4):
    """
    Time series cross validation for scikit learn models.
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    cv_scores = {'MAE': [], 'RMSE': [], 'MAPE': [], 'R2': []}
    
    for fold, (train_index, val_index) in enumerate(tscv.split(X), start=1):
        X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
        y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]
        
        # Train model on fold
        model.fit(X_train_fold, y_train_fold)
        y_pred_fold = model.predict(X_val_fold)
        
        # Calculate evaluation metrics
        mae = mean_absolute_error(y_val_fold, y_pred_fold)
        rmse = np.sqrt(mean_squared_error(y_val_fold, y_pred_fold))
        r2 = r2_score(y_val_fold, y_pred_fold)
        
        mask = y_val_fold > 0
        mape = np.mean(np.abs((y_val_fold[mask] - y_pred_fold[mask]) / y_val_fold[mask])) * 100
        
        # Save scores
        cv_scores['MAE'].append(mae)
        cv_scores['RMSE'].append(rmse)
        cv_scores['MAPE'].append(mape)
        cv_scores['R2'].append(r2)
        
        print(f"  Fold {fold} -> MAE: €{mae:,.0f} | MAPE: {mape:.1f}%")
        
    # Mean +- std scores
    cv_results = {
        'mean': {key: np.mean(values) for key, values in cv_scores.items()},
        'std': {key: np.std(values) for key, values in cv_scores.items()}
    }
    
    # Print results
    print(f" CV scores")
    print(f"  MAE   : €{cv_results['mean']['MAE']:,.0f}  (± €{cv_results['std']['MAE']:,.0f})")
    print(f"  RMSE  : €{cv_results['mean']['RMSE']:,.0f}  (± €{cv_results['std']['RMSE']:,.0f})")
    print(f"  MAPE  : {cv_results['mean']['MAPE']:.1f}%  (± {cv_results['std']['MAPE']:.1f}%)")
    print(f"  R2    : {cv_results['mean']['R2']:.3f}  (± {cv_results['std']['R2']:.3f})")
    
    # Retrain model on full dataset for test analysis
    model.fit(X, y)
    
    return cv_scores

In [0]:
def fy_revenue(y_test, y_pred):
    """
    Compares full year revenue of the test set to the predicted revenue
    """
    actual_annual_revenue = y_test.sum()
    predicted_annual_revenue = y_pred.sum()

    # Calculate absolute and percentage differences
    difference_euro = predicted_annual_revenue - actual_annual_revenue
    difference_percentage = (difference_euro / actual_annual_revenue) * 100

    print(f"Actual revenue 2025:    €{actual_annual_revenue:,.2f}")
    print(f"Predicted revenue 2025: €{predicted_annual_revenue:,.2f}")
    print(f"Difference:             €{difference_euro:,.2f} ({difference_percentage:+.2f}%)")

In [0]:
def compare_test_errors(y_true, y_pred_base, y_pred_new, base_name="SARIMAX", new_name=""):
    """
    Tests significance on MAE, RMSE and MAPE, the evaluation metrics
    """
    # MAE
    ae_base = np.abs(y_true - y_pred_base)
    ae_new = np.abs(y_true - y_pred_new)
    
    # RMSE
    se_base = (y_true - y_pred_base)**2
    se_new = (y_true - y_pred_new)**2
    
    # MAPE
    mask = y_true > 0
    ape_base = np.abs((y_true[mask] - y_pred_base[mask]) / y_true[mask]) * 100
    ape_new = np.abs((y_true[mask] - y_pred_new[mask]) / y_true[mask]) * 100

    # Tests
    _, p_mae = stats.wilcoxon(ae_base, ae_new)
    _, p_rmse = stats.wilcoxon(se_base, se_new)
    _, p_mape = stats.wilcoxon(ape_base, ape_new)

    # Metrics
    metrics = {
        'MAE':  (p_mae,  np.mean(ae_base),  np.mean(ae_new)),
        'RMSE': (p_rmse, np.mean(se_base),  np.mean(se_new)),
        'MAPE': (p_mape, np.mean(ape_base), np.mean(ape_new))
    }
    
    print(f"Test-set (2025) Significance: {base_name} vs {new_name}")
    for metric, (p_val, mean_base, mean_new) in metrics.items():
        if p_val < 0.05:
            better_model = new_name if mean_new < mean_base else base_name
            print(f" {metric:<4} : p = {p_val:.5f} -> Significant difference ({better_model} is better!)")
        else:
            print(f" {metric:<4} : p = {p_val:.5f} -> No significant differnece")

In [0]:
def compare_cv_metrics(cv_base, cv_new, base_name="SARIMAX", new_name="XGBoost"):
    """
    Compares cv scores on sigficant differences
    """
    print(f"Cross-Validation Significance: {base_name} vs {new_name}")
    
    for metric in ['MAE', 'RMSE', 'MAPE']:
        scores_base = cv_base[metric]
        scores_new = cv_new[metric]
        
        # Paired t-test
        _, p_val = stats.ttest_rel(scores_base, scores_new)
        
        # Voor print logica
        mean_base = np.mean(scores_base)
        mean_new = np.mean(scores_new)
        beter = new_name if mean_new < mean_base else base_name
        
        if p_val < 0.05:
            print(f" {metric:<4} : p = {p_val:.5f} -> {beter} is SIGNIFICANT better")
        else:
            print(f" {metric:<4} : p = {p_val:.5f} -> No significant difference")

In [0]:
def plot_shap_analysis(model, X_test, model_name="Model", sample_size=None):
    """
    SHAP analysis on model and prints top 10
    """
    # Sample dataset when sample_size is given
    if sample_size is not None and sample_size < len(X_test):
        X_eval = X_test.sample(n=sample_size, random_state=42)
        print(f"Data sample: {sample_size} selected rows from {len(X_test)}.")
    else:
        X_eval = X_test
        print(f"Use all rows: {len(X_test)}")

    
    all_ensemble_shap = []
    for i, sub_model in enumerate(model.estimators_):
        explainer = shap.TreeExplainer(sub_model)
        shap_values = explainer.shap_values(X_eval)
            
        # Handle different formats returned by TreeExplainer
        if hasattr(shap_values, "values"):
            shap_data = shap_values.values
        elif isinstance(shap_values, list):
            shap_data = shap_values[0]
        else:
            shap_data = shap_values
                
        all_ensemble_shap.append(shap_data)
        
    # Calculate the mathematical mean across all models in the ensemble
    final_shap_data = np.mean(all_ensemble_shap, axis=0)

    # Calculate feature importance
    mean_abs_shap = np.mean(np.abs(final_shap_data), axis=0)

    shap_importance_df = pd.DataFrame(
        {"Feature": X_eval.columns, "Mean_Absolute_SHAP_EUR": mean_abs_shap}
    ).sort_values(by="Mean_Absolute_SHAP_EUR", ascending=False)

    print(f"\nTop 10 important features ({model_name}):")
    print(shap_importance_df.head(10))

    # Generate plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(
        final_shap_data, X_eval, max_display=10, plot_type="bar", show=False
    )
    
    plt.title(f"SHAP Feature Importance ({model_name})")
    plt.tight_layout()
    
    # Save graph
    bestandsnaam = f"{model_name.lower().replace(' ', '_')}_shap_importance.pdf"
    plt.savefig(bestandsnaam, dpi=300, bbox_inches="tight")
    plt.show()
    
    return shap_importance_df

# Evaluation metrics - Private dataset

### Load dataset used for ML

In [0]:
pad_dataset = './dataset'

X_train = pd.read_csv(f'{pad_dataset}/X_train.csv')
X_test = pd.read_csv(f'{pad_dataset}/X_test.csv')
y_train = pd.read_csv(f'{pad_dataset}/y_train.csv')['totale_omzet']
y_test = pd.read_csv(f'{pad_dataset}/y_test.csv')['totale_omzet']
dates_train = pd.read_csv(f'{pad_dataset}/dates_train.csv')['rapportagedatum']
dates_test = pd.read_csv(f'{pad_dataset}/dates_test.csv')['rapportagedatum']

print("✅ Centrale dataset ingeladen!")

### Sarimax

In [0]:
pad_sarimax = './sarimax'

# load model
sarimax_model = SARIMAXResults.load(f'{pad_sarimax}/getraind_sarimax_model.pkl')

# dataframes
train_sarimax = pd.read_csv(f'{pad_sarimax}/train.csv')
test_sarimax  = pd.read_csv(f'{pad_sarimax}/test.csv')
results_df_sarimax = pd.read_csv(f'{pad_sarimax}/results_df.csv')

X_train_sarimax = pd.read_csv(f'{pad_sarimax}/X_train.csv')
X_test_sarimax  = pd.read_csv(f'{pad_sarimax}/X_test.csv')

dataframes = [train_sarimax, test_sarimax, X_train_sarimax, X_test_sarimax]

for df in dataframes:
    if 'rapportagedatum' in df.columns:
        df['rapportagedatum'] = pd.to_datetime(df['rapportagedatum'])

        df.set_index('rapportagedatum', inplace=True)

In [0]:
# Evaluations and plots of SARIMAX - Private
test_metrics_sarimax = evaluate(test_sarimax['totale_omzet'], test_sarimax['Voorspelde_Omzet'], 'Test 2025 – SARIMAX Baseline')
plot_predictions(test_sarimax['totale_omzet'], test_sarimax['Voorspelde_Omzet'], 'SARIMAX Baseline')
cv_sarimax = cross_validate_sarimax(train_sarimax['totale_omzet'], order=sarimax_model.model.order, seasonal_order=(sarimax_model.model.seasonal_order), n_splits=4)
fy_revenue(test_sarimax['totale_omzet'], test_sarimax['Voorspelde_Omzet'])

### XGBoost

In [0]:
pad_xgb = './xgb_resultaten'

# Load model
xgb_model = xgb.XGBRegressor()
xgb_model.load_model(f'{pad_xgb}/getraind_xgboost_model.json')

# Load data
test_xgb = pd.read_csv(f'{pad_xgb}/test_results.csv')
X_test_xgb = pd.read_csv(f'{pad_xgb}/X_test.csv')

# Restore index column from date
test_xgb['rapportagedatum'] = pd.to_datetime(test_xgb['rapportagedatum'])
test_xgb.set_index('rapportagedatum', inplace=True)
test_xgb.index.name = 'Datum'

In [0]:
# Random seeds
best_params_xgb = xgb_model.get_params()

seeds = [42, 123, 456, 789, 999]
xgb_estimators = []

for seed in seeds:
    params_for_seed = best_params_xgb.copy()
    params_for_seed['random_state'] = seed
    
    xgb_estimators.append((f'xgb_seed_{seed}', xgb.XGBRegressor(**params_for_seed)))

# Combine them in a VotingRegressor and fit on the training data
xgb_robust_model = VotingRegressor(estimators=xgb_estimators)
xgb_robust_model.fit(X_train, y_train)

# Overwrite necessary variables
xgb_model = xgb_robust_model
test_xgb['Voorspelde_omzet'] = xgb_model.predict(X_test_xgb)

In [0]:
evaluate(test_xgb['totale_omzet'], test_xgb['Voorspelde_Omzet'], 'Test 2025 - XGBoost')
fy_revenue(test_xgb['totale_omzet'], test_xgb['Voorspelde_Omzet'])

In [0]:
# Group data per day
test_xgb_daily = test_xgb.resample('D')[['totale_omzet', 'Voorspelde_Omzet']].sum()

test_metrics_xgb = evaluate(test_xgb_daily['totale_omzet'], test_xgb_daily['Voorspelde_Omzet'], 'Test 2025 - XGBoost (daily)')
plot_predictions(test_xgb_daily['totale_omzet'], test_xgb_daily['Voorspelde_Omzet'], 'Test 2025 - XGBoost (daily)')

In [0]:
# Cross validation 
cv_xgb = cross_validate_ml(xgb_model, X_train, y_train, n_splits=4)

In [0]:
# Cross validation per day
dates_train_dt = pd.to_datetime(dates_train)

train = X_train.copy()
train['totale_omzet'] = y_train
train['rapportagedatum'] = dates_train_dt

# Aggregate data per day
features = X_train.columns
agg_dict = {col: 'max' for col in features}
agg_dict['totale_omzet'] = 'sum'
train_daily = train.groupby('rapportagedatum').agg(agg_dict)

# Split data into X and Y
X_train_daily = train_daily[features]
y_train_daily = train_daily['totale_omzet']

# Start cross validation per day
cv_xgb_daily = cross_validate_ml(xgb_model, X_train_daily, y_train_daily, n_splits=4)

### Random Forest

In [0]:
pad_rf = './rf_resultaten'

# Load model
rf_model = joblib.load(f'{pad_rf}/getraind_rf_model.pkl')

# Load data
test_rf = pd.read_csv(f'{pad_rf}/test_results.csv')
X_test_rf = pd.read_csv(f'{pad_rf}/X_test.csv')

# Restore index column from date
test_rf['rapportagedatum'] = pd.to_datetime(test_rf['rapportagedatum'])
test_rf.set_index('rapportagedatum', inplace=True)
test_rf.index.name = 'Datum'

In [0]:
# Random seeds
best_params_rf = rf_model.get_params()

rf_estimators = []

for seed in seeds:
    # Copy the parameters and update only the random state
    params_for_seed = best_params_rf.copy()
    params_for_seed['random_state'] = seed
    
    # Add the model to the list of estimators
    rf_estimators.append((f'rf_seed_{seed}', RandomForestRegressor(**params_for_seed)))

# Combine them in a VotingRegressor and fit on the training data
rf_robust_model = VotingRegressor(estimators=rf_estimators)
rf_robust_model.fit(X_train, y_train)

# Overwrite necessary variables
rf_model = rf_robust_model
test_rf['Voorspelde_omzet'] = rf_model.predict(X_test_rf)

In [0]:
evaluate(test_rf['totale_omzet'], test_rf['Voorspelde_Omzet'], 'Test 2025 - Random Forest')
fy_revenue(test_rf['totale_omzet'], test_rf['Voorspelde_Omzet'])

In [0]:
# Group data per day
test_rf_daily = test_rf.resample('D')[['totale_omzet', 'Voorspelde_Omzet']].sum()

test_metrics_rf = evaluate(test_rf_daily['totale_omzet'], test_rf_daily['Voorspelde_Omzet'], 'Test 2025 - Random Forest (daily)')
plot_predictions(test_rf_daily['totale_omzet'], test_rf_daily['Voorspelde_Omzet'], 'Test 2025 - Random Forest (daily)')

In [0]:
# Cross validation
cv_rf = cross_validate_ml(rf_model, X_train, y_train, n_splits=4)

In [0]:
# Cross validation per day
cv_rf_daily = cross_validate_ml(rf_model, X_train_daily, y_train_daily, n_splits=4)

# Analysis

## Testing statistical difference on evaluation metrics (sub rq1)


### Wilcoxon on test metrics

In [0]:
# SARIMAX and XGB
compare_test_errors(test_sarimax['totale_omzet'], test_sarimax['Voorspelde_Omzet'], test_xgb_daily['Voorspelde_Omzet'], new_name="XGBoost")

In [0]:
# SARIMAX and RF
compare_test_errors(test_sarimax['totale_omzet'], test_sarimax['Voorspelde_Omzet'], test_rf_daily['Voorspelde_Omzet'], new_name="Random Forest")

In [0]:
# XGB and RF
compare_test_errors(test_xgb['totale_omzet'], test_xgb['Voorspelde_Omzet'], test_rf['Voorspelde_Omzet'], base_name="XGBoost", new_name="Random Forest")
compare_test_errors(test_xgb_daily['totale_omzet'], test_xgb_daily['Voorspelde_Omzet'], test_rf_daily['Voorspelde_Omzet'], base_name="XGBoost daily", new_name="Random Forest daily")

### Wilcoxon on cv scores

In [0]:
compare_cv_metrics(cv_sarimax, cv_xgb_daily, base_name="SARIMAX", new_name="XGBoost")

In [0]:
compare_cv_metrics(cv_sarimax, cv_rf_daily, base_name="SARIMAX", new_name="Random Forest")

In [0]:
compare_cv_metrics(cv_xgb, cv_rf, base_name="XGBoost", new_name="Random Forest")
compare_cv_metrics(cv_xgb_daily, cv_rf_daily, base_name="XGBoost daily", new_name="Random Forest daily")

## SHAP Analysis and feature importance (sub rq2)

### SHAP analysis

In [0]:
shap_df_xgb = plot_shap_analysis(xgb_model, X_test_xgb, model_name="XGBoost", sample_size=5000)

In [0]:
shap_df_rf = plot_shap_analysis(rf_model, X_test_rf, model_name="Random Forest", sample_size=5000)

### Feature importance


In [0]:
all_importances = []
for estimator in xgb_model.estimators_:
    all_importances.append(estimator.feature_importances_)

# Calculate the average feature importance
mean_importance = np.mean(all_importances, axis=0)

xgb_importance = pd.DataFrame({
    'Feature': X_test_xgb.columns,
    'Gain (%)': mean_importance * 100
}).sort_values(by='Gain (%)', ascending=False)

print("--- Top 10 Features: XGBoost (Gain) ---")
print(xgb_importance.head(10))

In [0]:
all_importances = []
for estimator in rf_model.estimators_:
    all_importances.append(estimator.feature_importances_)

# Calculate the average feature importance
mean_importance = np.mean(all_importances, axis=0)

xgb_importance = pd.DataFrame({
    'Feature': X_test_rf.columns,
    'Gain (%)': mean_importance * 100
}).sort_values(by='Gain (%)', ascending=False)

print("--- Top 10 Features: Random Forest (Gain) ---")
print(xgb_importance.head(10))

## Scenario / sensitivity analysis (sub rq3)

#### Scenario 1: Workforce capacity shock

In [0]:
xgb_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

capaciteit_kolommen = ['fte_bruto_contract', 'fte_netto_contract', 'fte_inzet', 'dienst_uren']

# Baseline
y_pred_baseline_xgb = xgb_model.predict(X_test_xgb)
totale_omzet_baseline_xgb = np.sum(y_pred_baseline_xgb)

y_pred_baseline_rf = rf_model.predict(X_test_xgb)
totale_omzet_baseline_rf = np.sum(y_pred_baseline_rf)

print(f"Scenario 1: Workforce capacity shock")
print(f"Baseline predicted revenue (2025) XGBoost : €{totale_omzet_baseline_xgb:,.0f}\n")
print(f"Baseline predicted revenue (2025) Random Forest : € {totale_omzet_baseline_rf:,.0f}\n")

# Test multiple shock percentages
krimp_percentages = [5, 10, 15]

for krimp in krimp_percentages:
    X_test_schok = X_test_xgb.copy()
    
    factor = 1 - (krimp / 100)
    
    for col in capaciteit_kolommen:
        if col in X_test_schok.columns:
            X_test_schok[col] = X_test_schok[col] * factor
            
    # XGBoost prediction
    y_pred_schok_xgb = xgb_model.predict(X_test_schok)
    totale_omzet_schok_xgb = np.sum(y_pred_schok_xgb)
    verschil_euro_xgb = totale_omzet_schok_xgb - totale_omzet_baseline_xgb
    verschil_procent_xgb = (verschil_euro_xgb / totale_omzet_baseline_xgb) * 100
    
    # RF prediction
    y_pred_schok_rf = rf_model.predict(X_test_schok)
    totale_omzet_schok_rf = np.sum(y_pred_schok_rf)
    verschil_euro_rf = totale_omzet_schok_rf - totale_omzet_baseline_rf
    verschil_procent_rf = (verschil_euro_rf / totale_omzet_baseline_rf) * 100

    # Print de analyse netjes uit
    print(f"🚨 Scenario: {krimp}% reduction in workforce capacity")
    print(f"  [XGBoost]       New revenue: €{totale_omzet_schok_xgb:,.0f} | Impact: €{verschil_euro_xgb:,.0f} ({verschil_procent_xgb:.2f}%)")
    print(f"  [Random Forest] New revenue: €{totale_omzet_schok_rf:,.0f} | Impact: €{verschil_euro_rf:,.0f} ({verschil_procent_rf:.2f}%)")
    print("-" * 70)

In [0]:
# Plot - use hard values so that I do not have to run the whole notebook again (and have slightly different outputs)
x_s1 = [0, -5, -10, -15]
rf_s1 = [0, -1.07, -2.00, -3.15]
xgb_s1 = [0, -1.43, -1.82, -4.24]

plt.figure(figsize=(8, 5))
sns.set_theme(style="whitegrid")

plt.plot(x_s1, rf_s1, marker='o', color='#1f77b4', linestyle='-', linewidth=2, markersize=8, label='Random Forest')
plt.plot(x_s1, xgb_s1, marker='s', color='#ff7f0e', linestyle='--', linewidth=2, markersize=8, label='XGBoost')

# Styling
plt.xlabel('Workforce Capacity Change (%)', fontsize=12, fontweight='bold')
plt.ylabel('Relative Financial Impact (%)', fontsize=12, fontweight='bold')
plt.title('Scenario 1: Revenue Impact of Workforce Capacity Shocks', fontsize=14, pad=15)
plt.legend(fontsize=11)
plt.gca().invert_xaxis() # As omdraaien zodat -15% rechts staat (krimptrend)

#### Scenario 2: Closing one unit

In [0]:
afdeling_kolom = "org_level_3_4247 - BGGZ HOPE Amstelland/Zuid-Kl"

mask_afdeling = X_test_xgb[afdeling_kolom] == 1
aantal_rijen = mask_afdeling.sum()

# Wat was de oorspronkelijke voorspelling voor *alleen* deze afdeling?
verwacht_afdeling_xgb = np.sum(y_pred_baseline_xgb[mask_afdeling])
verwacht_afdeling_rf = np.sum(y_pred_baseline_rf[mask_afdeling])

print(f"Scenario 2: Unit shutdown")
print(f"Afdeling : {afdeling_kolom}")
print(f"Predicted revenue (XGB)       : € {totale_omzet_baseline_xgb:,.0f}")
print(f"Predicted revenue (RF)        : € {totale_omzet_baseline_rf:,.0f}")
print(f"Predicted revenue for specific unit (XGB): € {verwacht_afdeling_xgb:,.0f}")
print(f"Predicted revenue for specific unit (RF) : € {verwacht_afdeling_rf:,.0f}\n")

X_test_schok = X_test_xgb.copy()
X_test_schok.loc[mask_afdeling, capaciteit_kolommen] = 0

y_pred_schok_xgb = xgb_model.predict(X_test_schok)
totale_omzet_schok_xgb = np.sum(y_pred_schok_xgb)
verschil_euro_xgb = totale_omzet_schok_xgb - totale_omzet_baseline_xgb
verschil_procent_xgb = (verschil_euro_xgb / totale_omzet_baseline_xgb) * 100

y_pred_schok_rf = rf_model.predict(X_test_schok)
totale_omzet_schok_rf = np.sum(y_pred_schok_rf)
verschil_euro_rf = totale_omzet_schok_rf - totale_omzet_baseline_rf
verschil_procent_rf = (verschil_euro_rf / totale_omzet_baseline_rf) * 100

print(f"[XGBoost]")
print(f"  New total revenue : € {totale_omzet_schok_xgb:,.0f}")
print(f"  Impact     : € {verschil_euro_xgb:,.0f} ({verschil_procent_xgb:.2f}%)")
print(f"  Difference compares for {np.abs(verschil_euro_xgb / verwacht_afdeling_xgb * 100):.1f}% with predicted revenue for specific unit")

print(f"[Random Forest]")
print(f"  New revenue : € {totale_omzet_schok_rf:,.0f}")
print(f"  Impact       : € {verschil_euro_rf:,.0f} ({verschil_procent_rf:.2f}%)")
print(f"  Difference compares for {np.abs(verschil_euro_rf / verwacht_afdeling_rf * 100):.1f}% with predicted revenue for specific unit")

#### Scenario 3: change bed capacity

In [0]:
bedden_kolom = 'available_beds'

print(f"Scenario 3: change bed capacity")
print(f"Predicted revenue (XGB) : € {totale_omzet_baseline_xgb:,.0f}")
print(f"Predicted revenue  : € {totale_omzet_baseline_rf:,.0f}\n")

bedden_variaties = [-10, -5, 5, 10]

for variatie in bedden_variaties:
    X_test_schok = X_test_xgb.copy()
    factor = 1 + (variatie / 100)
    
    if bedden_kolom in X_test_schok.columns:
        X_test_schok[bedden_kolom] = X_test_schok[bedden_kolom] * factor
        
    y_pred_schok_xgb = xgb_model.predict(X_test_schok)
    totale_omzet_schok_xgb = np.sum(y_pred_schok_xgb)
    verschil_euro_xgb = totale_omzet_schok_xgb - totale_omzet_baseline_xgb
    verschil_procent_xgb = (verschil_euro_xgb / totale_omzet_baseline_xgb) * 100
    
    y_pred_schok_rf = rf_model.predict(X_test_schok)
    totale_omzet_schok_rf = np.sum(y_pred_schok_rf)
    verschil_euro_rf = totale_omzet_schok_rf - totale_omzet_baseline_rf
    verschil_procent_rf = (verschil_euro_rf / totale_omzet_baseline_rf) * 100
    
    teken = "+" if variatie > 0 else ""
    print(f" Scenario: {teken}{variatie}% change in bed capacity")
    print(f"  [XGBoost]       New revenue: € {totale_omzet_schok_xgb:,.0f} | Impact: € {verschil_euro_xgb:,.0f} ({verschil_procent_xgb:.2f}%)")
    print(f"  [Random Forest] New revenue: € {totale_omzet_schok_rf:,.0f} | Impact: € {verschil_euro_rf:,.0f} ({verschil_procent_rf:.2f}%)")

In [0]:
x_s3 = [-10, -5, 0, 5, 10]
rf_s3 = [-2.19, -2.19, 0, 2.18, 2.18]
xgb_s3 = [-2.82, -2.82, 0, 2.86, 2.86]

plt.figure(figsize=(8, 5))
sns.set_theme(style="whitegrid")

plt.plot(x_s3, rf_s3, marker='o', color='#1f77b4', linestyle='-', linewidth=2, markersize=8, label='Random Forest')
plt.plot(x_s3, xgb_s3, marker='s', color='#ff7f0e', linestyle='--', linewidth=2, markersize=8, label='XGBoost')

# Styling
plt.xlabel('Bed Capacity Change (%)', fontsize=12, fontweight='bold')
plt.ylabel('Relative Financial Impact (%)', fontsize=12, fontweight='bold')
plt.title('Scenario 3: Revenue Impact of Bed Capacity Fluctuations', fontsize=14, pad=15)
plt.legend(fontsize=11)
plt.axhline(0, color='black', linewidth=1, linestyle='-', alpha=0.3) # Nul-lijn benadrukken
plt.axvline(0, color='black', linewidth=1, linestyle='-', alpha=0.3) # Nul-lijn benadrukken

# Evaluation metrics for public dataset

### Load public dataset and models

In [0]:
# Public dataset
pad_dataset = './dataset'

X_train_public = pd.read_csv(f'{pad_dataset}/X_train_public.csv')
X_test_public = pd.read_csv(f'{pad_dataset}/X_test_public.csv')
y_train_public = pd.read_csv(f'{pad_dataset}/y_train_public.csv')['TOTALE_OMZET']
y_test_public = pd.read_csv(f'{pad_dataset}/y_test_public.csv')['TOTALE_OMZET']
dates_train_public = pd.read_csv(f'{pad_dataset}/dates_train_public.csv')['DATUM']
dates_test_public = pd.read_csv(f'{pad_dataset}/dates_test_public.csv')['DATUM']

### SARIMAX

In [0]:
pad_sarimax = './sarimax'

sarimax_model_public = SARIMAXResults.load(f'{pad_sarimax}/sarimax_public.pkl')

train_sarimax_public = pd.read_csv(f'{pad_sarimax}/train_public.csv')
test_sarimax_public  = pd.read_csv(f'{pad_sarimax}/test_public.csv')
results_sarimax_public = pd.read_csv(f'{pad_sarimax}/results_sarimax_public.csv')
results_df_sarimax_public = pd.read_csv(f'{pad_sarimax}/results_df_public.csv')

X_train_sarimax_public = pd.read_csv(f'{pad_sarimax}/X_train_public.csv')
X_test_sarimax_public  = pd.read_csv(f'{pad_sarimax}/X_test_public.csv')

dataframes = [
    train_sarimax_public, 
    test_sarimax_public, 
    results_sarimax_public, 
    X_train_sarimax_public, 
    X_test_sarimax_public
]

for df in dataframes:
    if 'rapportagedatum' in df.columns:
        df['rapportagedatum'] = pd.to_datetime(df['rapportagedatum'])
        df.set_index('rapportagedatum', inplace=True)
    elif 'DATUM' in df.columns:
        df['DATUM'] = pd.to_datetime(df['DATUM'])
        df.set_index('DATUM', inplace=True)



In [0]:
# Evaluations and plots of SARIMAX - Public
test_metrics_sarimax_public = evaluate(results_sarimax_public['TOTALE_OMZET'], results_sarimax_public['Voorspelde_Omzet'], 'Test 2025 – SARIMAX Public')
plot_predictions(results_sarimax_public['TOTALE_OMZET'], results_sarimax_public['Voorspelde_Omzet'], 'Test 2025 – SARIMAX Public')
cv_sarimax_public = cross_validate_sarimax(train_sarimax_public['TOTALE_OMZET'], order=sarimax_model_public.model.order, seasonal_order=(sarimax_model_public.model.seasonal_order), n_splits=4)
fy_revenue(results_sarimax_public['TOTALE_OMZET'], results_sarimax_public['Voorspelde_Omzet'])

### XGBoost

In [0]:
pad_xgb = './xgb_resultaten'
pad_dataset = './dataset'

# Load model
xgb_model_public = xgb.XGBRegressor()
xgb_model_public.load_model(f'{pad_xgb}/xgboost_public.json')

# Predictions
test_xgb_public = pd.read_csv(f'{pad_xgb}/results_xgboost_public.csv')

test_xgb_public['rapportagedatum'] = pd.to_datetime(test_xgb_public['rapportagedatum'])
test_xgb_public.set_index('rapportagedatum', inplace=True)
test_xgb_public.index.name = 'Datum'

In [0]:
# Random seeds for more robustness
best_params_xgb_public = xgb_model_public.get_params()

seeds = [42, 123, 456, 789, 999]
xgb_estimators_public = []

# Init models per seed
for seed in seeds:
    params_for_seed = best_params_xgb_public.copy()
    params_for_seed['random_state'] = seed
    
    xgb_estimators_public.append((f'xgb_seed_{seed}', xgb.XGBRegressor(**params_for_seed)))

# Fit ensemble
xgb_robust_model_public = VotingRegressor(estimators=xgb_estimators_public)
xgb_robust_model_public.fit(X_train_public, y_train_public)

# Update predictions
xgb_model_public = xgb_robust_model_public
test_xgb_public['Voorspelde_Omzet'] = xgb_model_public.predict(X_test_public)

In [0]:
test_metrics_xgb_public = evaluate(test_xgb_public['totale_omzet'], test_xgb_public['Voorspelde_Omzet'], 'Test 2025 - XGBoost')
fy_revenue(test_xgb_public['totale_omzet'], test_xgb_public['Voorspelde_Omzet'])

In [0]:
# Group data per day
test_xgb_public_daily = test_xgb_public.resample('D')[['totale_omzet', 'Voorspelde_Omzet']].sum()

test_metrics_xgb_public_daily = evaluate(test_xgb_public_daily['totale_omzet'], test_xgb_public_daily['Voorspelde_Omzet'], 'Test 2025 - XGBoost (daily)')
plot_predictions(test_xgb_public_daily['totale_omzet'], test_xgb_public_daily['Voorspelde_Omzet'], 'Test 2025 - XGBoost (daily)')

In [0]:
# Cross validation 
cv_xgb_public = cross_validate_ml(xgb_model_public, X_train_public, y_train_public, n_splits=4)

In [0]:
# Convert public dates to datetime
dates_train_dt_public = pd.to_datetime(dates_train_public)

# Create public train dataframe
train_public = X_train_public.copy()
train_public['totale_omzet'] = y_train_public.values
train_public['rapportagedatum'] = dates_train_dt_public.values

# Aggregate data per day
features = X_train_public.columns
agg_dict = {col: 'max' for col in features}
agg_dict['totale_omzet'] = 'sum'
train_daily_public = train_public.groupby('rapportagedatum').agg(agg_dict)

# Split aggregated public data
X_train_daily_public = train_daily_public[features]
y_train_daily_public = train_daily_public['totale_omzet']

# Run cross validation on daily public data
cv_xgb_daily_public = cross_validate_ml(xgb_model_public, X_train_daily_public, y_train_daily_public, n_splits=4)

### Random Forest

In [0]:
pad_rf = './rf_resultaten'
pad_dataset = './dataset'

# Load public model
rf_model_public = joblib.load(f'{pad_rf}/rf_public.pkl')

# Load public test results
test_rf_public = pd.read_csv(f'{pad_rf}/results_rf_public.csv')

# Restore index column
test_rf_public['rapportagedatum'] = pd.to_datetime(test_rf_public['rapportagedatum'])
test_rf_public.set_index('rapportagedatum', inplace=True)
test_rf_public.index.name = 'Datum'

In [0]:
# Extract base params
best_params_rf_public = rf_model_public.get_params()

seeds = [42, 123, 456, 789, 999]
rf_estimators_public = []

# Init models per seed
for seed in seeds:
    params_for_seed = best_params_rf_public.copy()
    params_for_seed['random_state'] = seed
    
    rf_estimators_public.append((f'rf_seed_{seed}', RandomForestRegressor(**params_for_seed)))

# Fit ensemble
rf_robust_model_public = VotingRegressor(estimators=rf_estimators_public)
rf_robust_model_public.fit(X_train_public, y_train_public)

# Update predictions using the central X_test_public
rf_model_public = rf_robust_model_public
test_rf_public['Voorspelde_Omzet'] = rf_model_public.predict(X_test_public)

In [0]:
test_metrics_rf_public = evaluate(test_rf_public['totale_omzet'], test_rf_public['Voorspelde_Omzet'], 'Test 2025 - Random Forest')
fy_revenue(test_rf_public['totale_omzet'], test_rf_public['Voorspelde_Omzet'])

In [0]:
# Group data per day
test_rf_public_daily = test_rf_public.resample('D')[['totale_omzet', 'Voorspelde_Omzet']].sum()

test_metrics_rf_public_daily = evaluate(test_rf_public_daily['totale_omzet'], test_rf_public_daily['Voorspelde_Omzet'], 'Test 2025 - Random Forest (daily)')
plot_predictions(test_rf_public_daily['totale_omzet'], test_rf_public_daily['Voorspelde_Omzet'], 'Test 2025 - Random Forest (daily)')

In [0]:
# Cross validation
cv_rf_public = cross_validate_ml(rf_model_public, X_train_public, y_train_public, n_splits=4)

In [0]:
# Cross validation per day
cv_rf_daily_public = cross_validate_ml(rf_model_public, X_train_daily_public, y_train_daily_public, n_splits=4)

# Analysis - Public dataset

## Testing statistical difference on evaluation metrics (sub rq1)

### Wilcoxon on test metrics

In [0]:
# SARIMAX and XGB
compare_test_errors(results_sarimax_public['TOTALE_OMZET'], results_sarimax_public['Voorspelde_Omzet'], test_xgb_public_daily['Voorspelde_Omzet'], new_name="XGBoost")

In [0]:
# SARIMAX and RF
compare_test_errors(results_sarimax_public['TOTALE_OMZET'], results_sarimax_public['Voorspelde_Omzet'], test_rf_public_daily['Voorspelde_Omzet'], new_name="Random Forest")

In [0]:
# XGB and RF
compare_test_errors(test_xgb_public['totale_omzet'], test_xgb_public['Voorspelde_Omzet'], test_rf_public['Voorspelde_Omzet'], base_name="XGBoost", new_name="Random Forest")
compare_test_errors(test_xgb_public_daily['totale_omzet'], test_xgb_public_daily['Voorspelde_Omzet'], test_rf_public_daily['Voorspelde_Omzet'], base_name="XGBoost daily", new_name="Random Forest daily")

### Wilcoxon on cv scores

In [0]:
compare_cv_metrics(cv_sarimax_public, cv_xgb_daily_public, base_name="SARIMAX", new_name="XGBoost")

In [0]:
compare_cv_metrics(cv_sarimax_public, cv_rf_daily_public, base_name="SARIMAX", new_name="Random Forest")

In [0]:
compare_cv_metrics(cv_xgb_daily_public, cv_rf_daily_public, base_name="XGBoost daily", new_name="Random Forest daily")
compare_cv_metrics(cv_xgb_public, cv_rf_public, base_name="XGBoost", new_name="Random Forest")

## SHAP analysis and feature importances

### SHAP analysis

In [0]:
plot_shap_analysis(xgb_model_public, X_test_public, model_name="XGBoost")

In [0]:
plot_shap_analysis(rf_model_public, X_test_public, model_name="Random Forest")

### Feature importance

In [0]:
all_importances = []
for estimator in xgb_model_public.estimators_:
    all_importances.append(estimator.feature_importances_)

# Calculate the average feature importance
mean_importance = np.mean(all_importances, axis=0)

xgb_importance_public = pd.DataFrame({
    'Feature': X_test_public.columns,
    'Gain (%)': mean_importance * 100
}).sort_values(by='Gain (%)', ascending=False)

print("--- Top 10 Features: XGBoost (Gain) ---")
print(xgb_importance_public.head(10))

In [0]:
all_importances = []
for estimator in rf_model_public.estimators_:
    all_importances.append(estimator.feature_importances_)

# Calculate the average feature importance
mean_importance = np.mean(all_importances, axis=0)

rf_importance_public = pd.DataFrame({
    'Feature': X_test_public.columns,
    'Gain (%)': mean_importance * 100
}).sort_values(by='Gain (%)', ascending=False)

print("--- Top 10 Features: Random Forest (Gain) ---")
print(rf_importance_public.head(10))